# 18.3 Data cleaning and missing-value handling

Cleaning is not erasing mess; it is preserving signal while making assumptions explicit.

Missingness can be a measurement process, a segment signal, or a sampling intervention. This notebook compares imputation, indicators, and row dropping with a fixed downstream classifier.

Save a copy to Drive to edit.

In [ ]:

import math
import random

import matplotlib.pyplot as plt
import numpy as np
from sklearn.datasets import load_breast_cancer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler

np.random.seed(18)
random.seed(18)


def dataquality_ladder():
    """D1..D5 over real Breast Cancer with progressively worse data quality."""
    bc = load_breast_cancer()
    X0 = bc.data.astype(float)
    y0 = bc.target
    rng = np.random.default_rng(18)
    rungs = []

    rungs.append(("D1 clean", X0.copy(), y0.copy()))

    y2 = y0.copy()
    flip = rng.random(y2.shape) < 0.15
    y2[flip] = 1 - y2[flip]
    rungs.append(("D2 15% label noise", X0.copy(), y2))

    keep_pos = np.where(y0 == 1)[0]
    keep_neg = np.where(y0 == 0)[0][:30]
    idx = np.concatenate([keep_pos, keep_neg])
    rungs.append(("D3 class imbalance", X0[idx].copy(), y0[idx].copy()))

    X4 = X0 + rng.normal(0.0, X0.std(axis=0) * 0.5, size=X0.shape)
    rungs.append(("D4 heavy feature noise", X4, y0.copy()))

    X5 = X0.copy()
    holes = rng.random(X5.shape) < 0.2
    X5[holes] = np.nan
    col_means = np.nanmean(X5, axis=0)
    X5 = np.where(np.isnan(X5), col_means, X5)
    rungs.append(("D5 20% missing (mean-imputed)", X5, y0.copy()))

    return rungs


def split_scale(X, y, seed=0):
    x_train, x_test, y_train, y_test = train_test_split(
        X,
        y,
        test_size=0.35,
        random_state=seed,
        stratify=y,
    )
    scaler = StandardScaler()
    x_train = scaler.fit_transform(x_train)
    x_test = scaler.transform(x_test)
    return x_train, x_test, y_train, y_test


def fit_logistic(x_train, y_train, sample_weight=None):
    model = LogisticRegression(max_iter=2000, solver="lbfgs")
    model.fit(x_train, y_train, sample_weight=sample_weight)
    return model


def preview_ladder(rungs):
    for i, (name, X, y) in enumerate(rungs, start=1):
        counts = dict(zip(*np.unique(y, return_counts=True)))
        print(f"{i}. {name}: X={X.shape}, class_counts={counts}")
    print("sample rows", np.round(rungs[0][1][:3, :4], 3).tolist())


def plot_rungs_and_metric(rungs, rows, title, ylabel="accuracy"):
    fig, axes = plt.subplots(2, 5, figsize=(16, 6))
    metrics = [row["metric"] for row in rows]
    for ax, (name, X, y), metric in zip(axes[0], rungs, metrics):
        ax.scatter(X[:, 0], X[:, 1], c=y, s=12, cmap="coolwarm", alpha=0.65)
        ax.set_title(f"{name}\n{metric:.3f}")
        ax.set_xticks([])
        ax.set_yticks([])
    axes[1][0].plot(range(1, 6), metrics, marker="o")
    axes[1][0].set_title(title)
    axes[1][0].set_xlabel("data-quality rung")
    axes[1][0].set_ylabel(ylabel)
    axes[1][0].set_ylim(0.0, 1.05)
    for ax in axes[1][1:]:
        ax.axis("off")
    plt.tight_layout()
    plt.show()



## The concept, built once (D1)

Mean imputation uses $\mu_O=rac{1}{|O|}\sum_{i\in O}x_i$ over observed entries. Segment-aware imputation changes the observed set to the row's group, and a missingness indicator $m_i=\mathbf{1}[x_i	ext{ is missing}]$ preserves the process signal.

In [ ]:
def clean_matrix(X, add_indicator=True):
    X = np.asarray(X, dtype=float)
    missing = np.isnan(X)
    col_means = np.nanmean(X, axis=0)
    filled = np.where(missing, col_means, X)
    if add_indicator:
        return np.column_stack([filled, missing.astype(float)])
    return filled


def clean_and_train(X, y, seed=0, add_indicator=True):
    X_clean = clean_matrix(X, add_indicator=add_indicator)
    x_train, x_test, y_train, y_test = split_scale(X_clean, y, seed=seed)
    model = fit_logistic(x_train, y_train)
    preds = model.predict(x_test)
    return accuracy_score(y_test, preds)


def inject_informative_missingness(X, y, seed=18):
    rng = np.random.default_rng(seed)
    X_bad = X.copy().astype(float)
    positive_rows = np.where(y == 1)[0]
    chosen = rng.choice(positive_rows, size=min(170, len(positive_rows)), replace=False)
    X_bad[chosen, 0] = np.nan
    random_holes = rng.random(X_bad.shape) < 0.04
    X_bad[random_holes] = np.nan
    return X_bad

Assert the lesson's global fill, group fill, and complete-case label shift.

In [ ]:
values = np.array([10, 12, np.nan, 18, 20], dtype=float)
observed_mean = np.nanmean(values)
filled = np.where(np.isnan(values), observed_mean, values)
group_a = np.array([10, 12, np.nan], dtype=float)
group_b = np.array([30, 32, np.nan], dtype=float)
fill_a = np.nanmean(group_a)
fill_b = np.nanmean(group_b)
labels = np.array([0, 0, 1, 1, 1, 1])
complete = np.array([True, True, True, True, False, False])
full_rate = labels.mean()
complete_rate = labels[complete].mean()
print(round(observed_mean, 3), round(filled.mean(), 3), round(fill_a, 3), round(fill_b, 3))
print(round(full_rate, 3), round(complete_rate, 3))
assert round(observed_mean, 3) == 15.0
assert round(filled.mean(), 3) == 15.0
assert round(fill_a, 3) == 11.0
assert round(fill_b, 3) == 31.0
assert round(full_rate, 3) == 0.667
assert round(complete_rate, 3) == 0.5

## The dataset ladder

Every notebook uses the same real Breast Cancer base table and changes data quality only: clean, label-noisy, imbalanced, feature-noisy, then missing-and-imputed. The model stays fixed so movement in the curve is caused by the data.

In [ ]:
rungs = dataquality_ladder()
preview_ladder(rungs)

In [ ]:
rows = []
for name, X, y in rungs:
    X_use = inject_informative_missingness(X, y) if "D5" in name else X
    metric = clean_and_train(X_use, y, seed=1, add_indicator=True)
    rows.append({"name": name, "metric": metric})
    print(f"{name:28s} clean_indicator_accuracy={metric:.3f}")

In [ ]:
plot_rungs_and_metric(rungs, rows, "Accuracy after explicit cleaning")

## Pitfall on D5: dropping rows under informative missingness

Complete-case analysis silently reweights labels when missingness is concentrated among positives. The fix is imputation plus missingness indicators, followed by a label-rate check.

In [ ]:
name, X, y = rungs[-1]
X_bad = inject_informative_missingness(X, y)
complete_rows = ~np.isnan(X_bad).any(axis=1)
dropped_rate = y[complete_rows].mean()
full_rate = y.mean()
drop_acc = clean_and_train(X_bad[complete_rows], y[complete_rows], seed=2, add_indicator=False)
indicator_acc = clean_and_train(X_bad, y, seed=2, add_indicator=True)
print(f"full positive rate={full_rate:.3f} complete-case rate={dropped_rate:.3f}")
print(f"drop rows accuracy={drop_acc:.3f} impute+indicator accuracy={indicator_acc:.3f}")
assert abs(full_rate - dropped_rate) > 0.05
assert indicator_acc >= drop_acc - 0.05

## Evaluate it + Practice

- Metric: held-out accuracy after cleaning, plus label-rate shift under dropped rows; compare with the no-skill majority-class baseline.
- Cheap sanity check: the filled lesson vector keeps mean 15.000 but changes variance.
- Ablation: turn off missingness indicators and compare D5.
- Failure signals: complete-case label rate differs from the full label rate or missingness clusters by class.

Practice prompts:
1. Change one corruption level in `dataquality_ladder()` and predict the metric direction.


In [ ]:
# Your turn: change one corruption and rerun the ladder table.


2. Add one slice check that could catch a global-average pitfall.

In [ ]:
# Your turn: define a slice and compare its metric with the global metric.


3. Replace the fixed logistic model with another CPU-safe classifier and explain whether the data-quality ordering changed.

In [ ]:
# Your turn: try a second model without changing the data-cleaning method.
